In [1]:
import re
import os
import json

### For raw input

In [2]:
def clean_text(text: str) -> str:
    text = text.replace("&amp;", "&") # Remove HTML entities
    text = re.sub(r"\.{2,}", ".", text) # Remove excessive dots (e.g., "....")
    text = re.sub(r"\s+", " ", text) # Fix spacing
    text = re.sub(r"([.,!?])([A-Za-z])", r"\1 \2", text) # Add space after punctuation if missing
    text = text.replace("\\n", "\n") # Normalize line breaks
    text = text.strip() # Strip leading/trailing spaces
    return text


def normalize_sections(text: str) -> str:
    """
    Hybrid approach:
    1. Handle known fields explicitly
    2. Handle unknown fields dynamically
    """

    # Known fields (high confidence)
    known_fields = [
        "Job Title:", "Location:", "Country:",
        "Experience Required:", "Primary Skills:", "Secondary Skills:",
        "Job Description:", "Key Responsibilities:",
        "Skill Requirements:", "Additional Requirements:"
    ]

    for field in known_fields:
        text = text.replace(field, f"\n{field}")

    # Dynamic fallback (for unknown fields)
    text = re.sub(
        r"\s*((?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,3})\s*:)",
        r"\n\1",
        text
    )

    return text.strip()

def protect_urls(text: str):
    urls = re.findall(r"https?://\S+|www\.\S+|\b\S+\.com\b", text)
    
    url_map = {}
    for i, url in enumerate(urls):
        placeholder = f"__URL_{i}__"
        text = text.replace(url, placeholder)
        url_map[placeholder] = url
    
    return text, url_map

def restore_urls(text: str, url_map: dict):
    for placeholder, url in url_map.items():
        text = text.replace(placeholder, url)
    return text

def preprocess_raw_jd(raw_json: dict) -> str:
    """
    Main function to process raw JD
    """
    raw_text = raw_json.get("raw_jd", "")

    raw_text, url_map = protect_urls(raw_text)


    # Step 1: Clean noise
    cleaned = clean_text(raw_text)

    # Step 2: Normalize structure (light)
    normalized = normalize_sections(cleaned)

    normalized = restore_urls(normalized, url_map)


    # Step 3: Add instruction prompt
    final_input = f"""Industry: {raw_json.get("industry", "")}
{normalized}
"""
    

    return final_input


with open("jd_dataset/6/raw_jd.txt", "r") as f:
    data = json.load(f)

processed_input = preprocess_raw_jd(data)

print(processed_input)

Industry: Any Industry
Job Title: AWS
Senior Data Lead 
Location: N/A
Country:
Austria 
Experience Required: N/A
Primary Skills: SQL, Snowflake, DevOps Solutions, Python,
Amazon Glue 
Secondary Skills: Agile-Scrum (Digital)
Job Description: About HCL HCLTech is a global technology company, home to more than 223,400 people across 60 countries, delivering industry-leading capabilities centered around digital, engineering, cloud and AI, powered by a broad portfolio of technology services and products. We work with clients across all major verticals, providing industry solutions for Financial Services, Manufacturing, Life Sciences and Healthcare, Technology and Services, Telecom and Media, Retail and CPG, and Public Services. We deliver holistic services across industry verticals to leading enterprises, including 250 of the Fortune 500 and 650 of the Global 2000. Consolidated revenues as of 12 months ending June 2023 totaled $12.8 billion. To learn how we can supercharge progress for you, 

In [98]:
DATASET_PATH = "jd_dataset"

def process_dataset():
    for folder in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, folder)

        if os.path.isdir(folder_path):
            raw_file = os.path.join(folder_path, "raw_jd.txt")

            if os.path.exists(raw_file):
                try:
                    with open(raw_file, "r", encoding="utf-8") as f:
                        raw_text = json.load(f)

                    cleaned = preprocess_raw_jd(raw_text)

                    # Save cleaned output
                    output_file = os.path.join(folder_path, "cleaned_raw.txt")

                    with open(output_file, "w", encoding="utf-8") as f:
                        f.write(cleaned)

                    print(f"Processed folder {folder}")

                except Exception as e:
                    print(f"Error in folder {folder}: {e}")


if __name__ == "__main__":
    process_dataset()

Processed folder 114
Processed folder 238
Processed folder 186
Processed folder 44
Processed folder 82
Processed folder 14
Processed folder 109
Processed folder 176
Processed folder 252
Processed folder 184
Processed folder 178
Processed folder 22
Processed folder 194
Processed folder 210
Processed folder 70
Processed folder 61
Processed folder 126
Processed folder 160
Processed folder 259
Processed folder 158
Processed folder 244
Processed folder 173
Processed folder 24
Processed folder 257
Processed folder 190
Processed folder 86
Processed folder 62
Processed folder 207
Processed folder 60
Processed folder 75
Processed folder 165
Processed folder 248
Processed folder 96
Processed folder 25
Processed folder 195
Processed folder 182
Processed folder 168
Processed folder 65
Processed folder 117
Processed folder 59
Processed folder 288
Processed folder 192
Processed folder 111
Processed folder 150
Processed folder 139
Processed folder 205
Processed folder 202
Processed folder 218
Process

### For Enhanced output

In [ ]:
def clean_line(line: str) -> str:
    # Remove markdown bold and extra spaces
    line = line.replace("**", "").strip()
    return line


def parse_enhanced_md(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    data = {
        "job_title": "",
        "location": "",
        "industry": "",
        "responsibilities": [],
        "requirements": [],
        "qualifications": [],
        "experience": [],
        "other_requirements": []
    }

    current_section = None
    current_subsection = None

    for line in lines:
        line = clean_line(line)

        if not line:
            continue

        # 🔹 Detect main sections
        if line.startswith("##"):
            header = line.replace("##", "").strip().lower()

            if "job title" in header:
                current_section = "job_title"
            elif "location" in header:
                current_section = "location"
            elif "industry" in header:
                current_section = "industry"
            elif "responsibilities" in header:
                current_section = "responsibilities"
            elif "skill" in header:
                current_section = "requirements"
            elif "other" in header:
                current_section = "other_requirements"
            else:
                current_section = None

            current_subsection = None
            continue

        # 🔹 Detect subsections inside Skill Requirements
        if current_section == "requirements":
            if ":" in line and not line.startswith("-"):
                sub = line.lower()

                if "must have" in sub:
                    current_subsection = "requirements"
                    continue
                elif "educational" in sub:
                    current_subsection = "qualifications"
                    continue
                elif "experience" in sub:
                    current_subsection = "experience"
                    continue
                elif "professional attributes" in sub:
                    current_subsection = "requirements"
                    continue

        # 🔹 Handle content
        if current_section == "job_title":
            data["job_title"] = line

        elif current_section == "location":
            data["location"] = line

        elif current_section == "industry":
            data["industry"] = line

        elif current_section == "responsibilities":
            if line.startswith("-"):
                data["responsibilities"].append(line[1:].strip())
            else:
                data["responsibilities"].append(line)

        elif current_section == "requirements":
            if line.startswith("-"):
                item = line[1:].strip()

                if current_subsection == "qualifications":
                    data["qualifications"].append(item)
                elif current_subsection == "experience":
                    data["experience"].append(item)
                else:
                    data["requirements"].append(item)

        elif current_section == "other_requirements":
            if line.startswith("-"):
                data["other_requirements"].append(line[1:].strip())
            else:
                data["other_requirements"].append(line)

    return data

In [149]:
parsed = parse_enhanced_md("jd_dataset/256/enhanced_job_description.md")
parsed

{'job_title': 'React Developer (Next.js & Tailwind)',
 'location': 'N/A',
 'industry': 'Information Technology & Services',
 'responsibilities': ['Develop, maintain, and optimize responsive web applications using React.js, Next.js, and Tailwind CSS.',
  'Collaborate with UI/UX designers and backend developers to implement seamless, scalable, and high-performance user interfaces.',
  'Translate business and technical requirements into well-architected front-end solutions.',
  'Ensure code quality, maintainability, and performance through code reviews, unit testing, and best practices.',
  'Troubleshoot, debug, and resolve technical issues across browsers and devices.',
  'Participate in Agile ceremonies, actively contributing to sprint planning, estimation, and retrospectives.',
  'Stay up-to-date with emerging technologies, frameworks, and industry trends to continuously improve development processes.'],
 'requirements': ['Bachelor’s degree in Computer Science, Information Technology, 

In [153]:
DATASET_PATH = "jd_dataset"

def process_dataset():
    for folder in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, folder)

        if os.path.isdir(folder_path):
            md_file = os.path.join(folder_path, "enhanced_job_description.md")

            if os.path.exists(md_file):
                try:
                    parsed = parse_enhanced_md(md_file)

                    # Save cleaned output as JSON
                    output_file = os.path.join(folder_path, "enhanced_structured.json")

                    with open(output_file, "w", encoding="utf-8") as f:
                        json.dump(parsed, f, indent=2, ensure_ascii=False)

                    print(f"Processed folder {folder}")

                except Exception as e:
                    print(f"Error in folder {folder}: {e}")


if __name__ == "__main__":
    process_dataset()

Processed folder 114
Processed folder 238
Processed folder 186
Processed folder 44
Processed folder 82
Processed folder 14
Processed folder 109
Processed folder 176
Processed folder 252
Processed folder 184
Processed folder 178
Processed folder 22
Processed folder 194
Processed folder 210
Processed folder 70
Processed folder 61
Processed folder 126
Processed folder 160
Processed folder 259
Processed folder 158
Processed folder 244
Processed folder 173
Processed folder 24
Processed folder 257
Processed folder 190
Processed folder 86
Processed folder 62
Processed folder 207
Processed folder 60
Processed folder 75
Processed folder 165
Processed folder 248
Processed folder 96
Processed folder 25
Processed folder 195
Processed folder 182
Processed folder 168
Processed folder 65
Processed folder 117
Processed folder 59
Processed folder 288
Processed folder 192
Processed folder 111
Processed folder 150
Processed folder 139
Processed folder 205
Processed folder 202
Processed folder 218
Process

In [ ]:
DATASET_PATH = "jd_dataset"
OUTPUT_FILE = "final_dataset.json"

dataset = []

def build_dataset():
    for folder in os.listdir(DATASET_PATH):
        folder_path = os.path.join(DATASET_PATH, folder)

        if os.path.isdir(folder_path):
            raw_file = os.path.join(folder_path, "cleaned_raw.txt")
            enhanced_file = os.path.join(folder_path, "enhanced_structured.json")

            if os.path.exists(raw_file) and os.path.exists(enhanced_file):
                try:
                    # Load raw
                    with open(raw_file, "r", encoding="utf-8") as f:
                        raw_text = f.read().strip()

                    # Load structured output
                    with open(enhanced_file, "r", encoding="utf-8") as f:
                        structured = json.load(f)

                    # Convert output to string (important)
                    output_text = json.dumps(structured, ensure_ascii=False)

                    # Final pair
                    sample = {
                        "input": raw_text,
                        "output": output_text
                    }

                    dataset.append(sample)

                except Exception as e:
                    print(f"Error in {folder}: {e}")

    # Save dataset
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print(f"Dataset created with {len(dataset)} samples")


if __name__ == "__main__":
    build_dataset()

Dataset created with 289 samples


In [3]:
from sklearn.model_selection import train_test_split
import json

with open("final_dataset.json", "r") as f:
    data = json.load(f)

train, val = train_test_split(data, test_size=0.2, random_state=42)

with open("train.json", "w") as f:
    json.dump(train, f, indent=2)

with open("val.json", "w") as f:
    json.dump(val, f, indent=2)

print(len(train), len(val))

231 58
